# 05. Multi-band photometry in one call

Measure a 50-arcsec field with `run_forced_photometry`. The driver loads the
catalog, images and PSFs, fits VIS profiles, then fits their fluxes in the
Euclid Y, J and H bands, unWISE W1 and W2, and IRAC channels 1 and 2. It
returns one catalog with every band and diagnostic metadata.

Notebooks [01](01_nisp_forced_photometry.ipynb),
[02](02_wise_forced_photometry.ipynb) and [03](03_irac_forced_photometry.ipynb)
explain each step and its limits. The source morphology is shared across
bands; estimated errors do not include every source of model or blend
uncertainty.

In [ ]:
DATA_ACCESS = "DR1"  ## Choose "DR1" or "Q1".
import euclid_phot as ep
ep.set_data_access(DATA_ACCESS)

import numpy as np

DATA_DIR = ep.get_data_dir()
RA, DEC, SIZE_ARCSEC = 269.48, 67.30, 50.0
EUCLID_BANDS = ("Y", "J", "H")
WISE_BANDS = ("W1", "W2")       ## () skips unWISE; needs the WISE dependencies
IRAC_BANDS = ("IRAC1", "IRAC2")  ## () skips IRAC


## Run the field

The call below takes the field centre and size, says where the source
models come from, and lists the bands to measure. The source models are
chosen on VIS by the model-selection tree (`"tree"`); `"prior"` would
start from the MER classifications and shapes instead. Each source is
rendered with the nearest GRID-PSF sample inside the joint fit, on the
MER pixel grid as delivered. For IRAC, `irac_options` picks the mosaics:
`"dawn"` covers the Euclid deep fields and `"seip"`, the default, has
wider coverage.

The first call on a new field downloads its inputs; later calls read the
cache. A field that crosses a MER tile boundary has to be fitted tile by
tile.


In [3]:
result = ep.run_forced_photometry(
    RA, DEC, SIZE_ARCSEC,
    prior={"band": "VIS", "objects": "mer", "model_selection": "tree"},
    target_bands={"euclid": EUCLID_BANDS, "wise": WISE_BANDS, "irac": IRAC_BANDS},
    irac_options={"mosaic_backend": "dawn"},
    psf_product="grid", persource_psf=True,
    mask_bright_stars=True, with_flag=True, calibrate_errors=True,
    data_dir=DATA_DIR, n_workers=4, verbose=True,
    force_download=True,
)


[1/7] query MER catalog at (269.4800, 67.3000)


/Users/ianlin/Desktop/euclid-forced-photometry-dev/src/euclid_phot/pipeline.py:529: UserWarning: DR1 morphology_table is not configured in src/conf.json; Sersic shape columns are masked. The existing source builder uses its isophotal/exponential fallback; tree-selected shapes are still fitted to the VIS pixels.
  result.mer_cat = query_mer_catalog(target_ra, target_dec, half_size_deg)


[2/7] discover mosaics for VIS, Y, J, H (lazy; only on a cache miss)
[3/7] fetch cutouts: VIS, Y, J, H
[4/7] build per-band PSFs (GRID-PSF, radius=60")
[5/7] build Tractor image + source models on VIS
[5a] STARSIGNAL bright-star mask: 7958 pixels (3.18%) excluded from the VIS fit
[6/7] VIS fit (free, forced-photometry)
[7/7] NISP forced phot: Y, J, H
[7a] calibrate flux errors (empty-position point-source fits)
  VIS: error inflation x1.00 (empty-position point-source, n=199)
  Y: error inflation x1.12 (empty-position point-source, n=200)
  J: error inflation x1.26 (empty-position point-source, n=200)
  H: error inflation x1.21 (empty-position point-source, n=200)
[+] WISE forced phot: W1, W2
[+] IRAC forced phot: IRAC1, IRAC2
  IRAC1: mosaic and effective PRF (dawn)


  IRAC1: 131 sources fitted, chi inflation 1.72
  IRAC2: mosaic and effective PRF (dawn)


  IRAC2: 131 sources fitted, chi inflation 1.00


## Inspect the catalog

The catalog comes from `result.to_table()`: one row per source, with the
flux density in microJansky and its error in every band, and an AB
magnitude where the flux is positive. A flux the fit could not constrain,
or that failed the flux guard, is NaN. `reliable` is False when the
source is near a bright star, on masked pixels, or at the cutout edge.

The table metadata records the PSF treatment, the uncertainty definitions
and the calibration factors. Extinction-corrected magnitudes use the
available foreground dust estimate; the measured fluxes themselves are
unchanged.


In [4]:
catalog = result.to_table()
print(f"{len(catalog)} sources; {len(catalog.colnames)} columns")
print("Euclid error inflation:", catalog.meta.get("error_inflation", {}))
print("PSF treatment:", catalog.meta["psf_mode"])
columns = ["object_id", "model", "reliable"] + [
    f"flux_{band}_ujy" for band in ("VIS", *EUCLID_BANDS, *WISE_BANDS, *IRAC_BANDS)]
rows = np.flatnonzero(np.isfinite(catalog["flux_VIS_ujy"]))[:8]
catalog[columns][rows].pprint_all()


131 sources; 65 columns
Euclid error inflation: {'VIS': 1.0, 'Y': 1.1204, 'J': 1.2637, 'H': 1.2091}
PSF treatment: nearest-source-position
     object_id         model    reliable      flux_VIS_ujy          flux_Y_ujy           flux_J_ujy           flux_H_ujy          flux_W1_ujy         flux_W2_ujy        flux_IRAC1_ujy      flux_IRAC2_ujy   
                                                  uJy                  uJy                  uJy                  uJy                  uJy                 uJy                 uJy                 uJy         
------------------- ----------- -------- --------------------- -------------------- -------------------- -------------------- ------------------- ------------------- ------------------- --------------------
2694934373672995764   ExpGalaxy     True  0.008980155381838367 -0.08688472374458842  0.10530986487171999 -0.03356929745587856 -182.44526657082304   71.73491719313384  0.8560723048654044  -0.9774635213508384
2694885408673025495 PointSource  

Save the table with `catalog.write("photometry.csv", overwrite=True)`, or
as `.ecsv` to keep the units and metadata. For another field,
change `RA`, `DEC` and `SIZE_ARCSEC`, then rerun the call and catalog cells.
